# PRIMA PROVA SVM

---

Leggo il nostro dict in `.pkl`

In [18]:
import pickle as pkl

with open('data/ottani_NIR.pkl', 'rb') as f:
    dizionario = pkl.load(f)
    
# dati train
train_ones = dizionario['train']['NIR']['ones']
train_zeros = dizionario['train']['NIR']['zeros']
train_labels = dizionario['train']['labels']
ottani_train_ones = dizionario['train']['octane']['ones']
ottani_train_zeros = dizionario['train']['octane']['zeros']

# dati test
test_ones = dizionario['test']['NIR']['ones']
test_zeros = dizionario['test']['NIR']['zeros']
test_labels = dizionario['test']['labels']


In [19]:
import numpy as np

train_x = np.concatenate((train_zeros, train_ones))
test_x = np.concatenate((test_zeros,test_ones))

ottani_train = np.concatenate((ottani_train_zeros, ottani_train_ones))


<blank>

## SMOOTHING 

---

provo a filtrare i dati con Savitzky-Golay

In [3]:
from scipy.signal import savgol_filter

window_size = 11
poly_order = 3

for i in range(train_x.shape[0]):
    train_x[i] = savgol_filter(train_x[i], window_size, poly_order)

for j in range(test_x.shape[0]):
    test_x[j] = savgol_filter(test_x[j], window_size, poly_order)

<blank>

## SVM

---

In [4]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn import svm

import pandas as pd

In [ ]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=10, random_state=42)

N_COMPONENTS_OPTIONS = [2, 5, None]
C_OPTIONS = [0.00001, 0.1, 1, 10]
KERNEL_OPTIONS = ["linear", "poly", "rbf"] 
GAMMA_OPTIONS = ['scale', 'auto', 0.01, 1]

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling (# NOTE: va inserito uno scaler placeholder)
    ("scaling", StandardScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=42)),
    
    # Step 3: Classificatore
    ("classify", svm.SVC(random_state=42)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare diversi oggetti Scaler
    "scaling": [StandardScaler(), MinMaxScaler(), RobustScaler()],
    
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per provare parametri del classificatore
    "classify__C": C_OPTIONS,
    
    # Per provare diversi kernel
    "classify__kernel": KERNEL_OPTIONS,
    
    # Gamma dei kernel
    "classify__gamma": GAMMA_OPTIONS,
},
{
    # BLOCCO 2: Senza PCA
    # Per provare diversi oggetti Scaler
    "scaling": [StandardScaler(), MinMaxScaler(), RobustScaler()],
    
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim": ['passthrough'], 
    
    # Per provare parametri del classificatore
    "classify__C": C_OPTIONS,
    
    # Per provare diversi kernel
    "classify__kernel": KERNEL_OPTIONS,
    
    # Gamma dei kernel
    "classify__gamma": GAMMA_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
    'score': 'accuracy',
    'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
    )

# 4. Training e Validation (su Segnale B) #
grid.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(test_x, test_labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale:.4f}")

# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)

La miglior configurazione: {'classify__C': 10, 'classify__gamma': 'scale', 'classify__kernel': 'linear', 'reduce_dim__n_components': 5, 'scaling': MinMaxScaler()}
Fornisce accuracy in validation: 0.9792
Risultato sul set indipendente (Segnale A): 1.0000


In [23]:
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_reduce_dim__n_components',
    'param_scaling', 
    'param_classify__C', 
    'param_classify__kernel',
    'param_classify__gamma',
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')
renamed = analysis.rename(columns={
    'param_scaling': 'scaler',
    'param_reduce_dim__n_components': 'pca_n_components', 
    'param_classify__C': 'C', 
    'param_classify__kernel': 'kernel',
    'param_classify__gamma': 'gamma',
    'mean_test_score': 'mean_score', 
    'std_test_score': 'std_score', 
    'mean_test_sensitivity': 'mean_sensitivity',
    'rank_test_score': 'rank_score',
})

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
renamed.head(20)

Numero totale di configurazioni provate: 576


,pca_n_components,scaler,C,kernel,gamma,mean_score,std_score,mean_sensitivity,rank_score
409,5,MinMaxScaler(),10.0,linear,1,0.979167,0.046585,0.958333,1
355,5,MinMaxScaler(),10.0,linear,auto,0.979167,0.046585,0.958333,1
328,5,MinMaxScaler(),10.0,linear,scale,0.979167,0.046585,0.958333,1
382,5,MinMaxScaler(),10.0,linear,0.01,0.979167,0.046585,0.958333,1
300,5,StandardScaler(),1.0,linear,1,0.975000,0.050000,0.950000,5
302,5,RobustScaler(),1.0,linear,1,0.975000,0.050000,0.950000,5
327,5,StandardScaler(),10.0,linear,scale,0.975000,0.050000,0.950000,5
408,5,StandardScaler(),10.0,linear,1,0.975000,0.050000,0.950000,5
329,5,RobustScaler(),10.0,linear,scale,0.975000,0.050000,0.950000,5
383,5,RobustScaler(),10.0,linear,0.01,0.975000,0.050000,0.950000,5


In [26]:
results_df.to_pickle("results/results-svm-savgol-GridSearch.pkl")

In [ ]:
results_df

## fine tuning

Da tenere sono:
- PCA 5 components
- normalizzazione MinMaxScaler
- kernel linear
- C 10 -> provare >10

In [27]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=10, random_state=42)

N_COMPONENTS_OPTIONS = [5, 19, 37]
C_OPTIONS = [5, 7, 10, 12, 15]
KERNEL_OPTIONS = ["linear"] 
GAMMA_OPTIONS = ['auto']

# 1. Definizione Pipeline #
pipe_fine = Pipeline([
    # Step 1: Scaling (# NOTE: va inserito uno scaler placeholder)
    ("scaling", MinMaxScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=42)),
    
    # Step 3: Classificatore
    ("classify", svm.SVC(random_state=42)) 
])

# 2. Definizione griglia dei parametri #
param_grid_fine = [
{   
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per provare parametri del classificatore
    "classify__C": C_OPTIONS,
    
    # Per provare diversi kernel
    "classify__kernel": KERNEL_OPTIONS,
    
    # Gamma dei kernel
    "classify__gamma": GAMMA_OPTIONS,
},
{
    # BLOCCO 2: Senza PCA
    
    # Per provare parametri specifici di uno step (PCA)
    "reduce_dim": ['passthrough'], 
    
    # Per provare parametri del classificatore
    "classify__C": C_OPTIONS,
    
    # Per provare diversi kernel
    "classify__kernel": KERNEL_OPTIONS,
    
    # Gamma dei kernel
    "classify__gamma": GAMMA_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid_fine = GridSearchCV(
    pipe_fine, 
    param_grid=param_grid_fine, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
    'score': 'accuracy',
    'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
    )

# 4. Training e Validation (su Segnale B) #
grid_fine.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid_fine.best_params_}")
print(f"Fornisce accuracy in validation: {grid_fine.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale_fine = grid_fine.score(test_x, test_labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale_fine:.4f}")

# Conversione dei risultati in DataFrame
results_df_fine = pd.DataFrame(grid_fine.cv_results_)

La miglior configurazione: {'classify__C': 5, 'classify__gamma': 'auto', 'classify__kernel': 'linear', 'reduce_dim__n_components': 5}
Fornisce accuracy in validation: 0.9812
Risultato sul set indipendente (Segnale A): 1.0000


In [28]:
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df_fine.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show_fine = [
    'param_reduce_dim__n_components',
    'param_classify__C',
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'std_test_sensitivity', 
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis_fine = results_df_fine[columns_to_show_fine].sort_values('rank_test_score')
renamed_fine = analysis_fine.rename(columns={
    'param_reduce_dim__n_components': 'pca_n_components', 
    'param_classify__C': 'C', 
    'mean_test_score': 'mean_score', 
    'std_test_score': 'std_score', 
    'mean_test_sensitivity': 'mean_sensitivity',
    'std_test_sensitivity': 'std_sensitivity',
    'rank_test_score': 'rank_score',
})

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
renamed_fine.head(20)

Numero totale di configurazioni provate: 20


,pca_n_components,C,mean_score,std_score,mean_sensitivity,std_sensitivity,rank_score
0,5.0,5,0.98125,0.044634,0.966667,0.084984,1
12,5.0,15,0.98125,0.044634,0.966667,0.084984,1
6,5.0,10,0.98125,0.044634,0.966667,0.084984,1
9,5.0,12,0.98125,0.044634,0.966667,0.084984,1
3,5.0,7,0.98125,0.044634,0.966667,0.084984,1
4,19.0,7,0.95625,0.059621,0.991667,0.044876,6
5,37.0,7,0.95625,0.059621,0.991667,0.044876,6
2,37.0,5,0.95625,0.059621,0.991667,0.044876,6
7,19.0,10,0.95625,0.059621,0.991667,0.044876,6
8,37.0,10,0.95625,0.059621,0.991667,0.044876,6


In [29]:
results_df_fine.to_pickle("results/results-svm-savgol-fine-GridSearch.pkl")

In [22]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# 1. Recuperiamo le 20 migliori configurazioni dal dataframe dei risultati
results_df = pd.DataFrame(grid.cv_results_)
top_20_indices = results_df.nsmallest(20, 'rank_test_score').index

# Vettore per tenere il conto degli errori (lungo 48)
error_counts = np.zeros(len(train_x))

# 2. Usiamo uno split singolo per permettere il cross_val_predict
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

print("Testo le 20 migliori configurazioni sui fold...")

# 3. Iteriamo sui top 20 modelli
for idx in top_20_indices:
    # Estraiamo il dizionario coi parametri di questo specifico modello
    params = results_df.loc[idx, 'params']
    
    # Cloniamo la pipeline vuota e le appiccichiamo i parametri top
    model = clone(pipe)
    model.set_params(**params)
    
    # Ora cross_val_predict funziona!
    y_pred = cross_val_predict(model, train_x, train_labels, cv=skf)
    
    # Aggiungiamo +1 ogni volta che questo modello sbaglia un campione
    error_counts += (y_pred != train_labels)

# 4. Analizziamo il bollettino di guerra
campioni_problematici = np.where(error_counts > 0)[0]

print("\n--- RISULTATI DEI CAMPIONI CRITICI ---")
for i in campioni_problematici:
    valore_reale = ottani_train[i] # Sostituisci con la tua variabile degli ottani continui
    volte_sbagliato = int(error_counts[i])
    print(f"Indice {i:2d} | Ottani reali: {valore_reale:.2f} | Sbagliato da {volte_sbagliato}/20 modelli top")


Testo le 20 migliori configurazioni sui fold...

--- RISULTATI DEI CAMPIONI CRITICI ---
Indice 26 | Ottani reali: 88.00 | Sbagliato da 20/20 modelli top
Indice 30 | Ottani reali: 88.25 | Sbagliato da 20/20 modelli top


In [25]:
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classify__C,param_classify__gamma,param_classify__kernel,param_reduce_dim__n_components,param_scaling,param_reduce_dim,...,split23_test_sensitivity,split24_test_sensitivity,split25_test_sensitivity,split26_test_sensitivity,split27_test_sensitivity,split28_test_sensitivity,split29_test_sensitivity,mean_test_sensitivity,std_test_sensitivity,rank_test_sensitivity
0,0.013845,0.012718,0.008067,0.011684,0.00001,scale,linear,2,StandardScaler(),NaN,...,0.75,0.75,0.50,1.0,0.75,1.0,0.75,0.758333,0.198781,426
1,0.006160,0.007004,0.003697,0.002811,0.00001,scale,linear,2,MinMaxScaler(),NaN,...,0.75,0.75,0.50,1.0,0.75,1.0,0.75,0.758333,0.198781,426
2,0.030779,0.008847,0.003112,0.001576,0.00001,scale,linear,2,RobustScaler(),NaN,...,0.75,0.75,0.50,1.0,0.75,1.0,0.75,0.766667,0.181812,418
3,0.003386,0.001245,0.002413,0.001125,0.00001,scale,linear,5,StandardScaler(),NaN,...,0.75,1.00,0.50,1.0,0.75,1.0,0.75,0.800000,0.175594,375
4,0.003398,0.001408,0.002103,0.000807,0.00001,scale,linear,5,MinMaxScaler(),NaN,...,0.75,1.00,0.50,1.0,0.75,1.0,0.75,0.791667,0.183523,391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,0.001590,0.000448,0.002322,0.000730,10.00000,1,poly,NaN,MinMaxScaler(),passthrough,...,0.75,1.00,0.75,1.0,1.00,1.0,0.75,0.925000,0.131498,224
572,0.029925,0.002860,0.002515,0.000676,10.00000,1,poly,NaN,RobustScaler(),passthrough,...,1.00,1.00,0.75,1.0,1.00,1.0,1.00,0.941667,0.105738,198
573,0.002128,0.000617,0.002771,0.000954,10.00000,1,rbf,NaN,StandardScaler(),passthrough,...,1.00,1.00,1.00,1.0,1.00,1.0,1.00,1.000000,0.000000,1
574,0.002027,0.000527,0.002881,0.000951,10.00000,1,rbf,NaN,MinMaxScaler(),passthrough,...,1.00,1.00,1.00,1.0,1.00,1.0,0.75,0.941667,0.123884,198
